# Homework 5 — Fine-Tuning LLM using LoRA



In [1]:
# Step 0 — Personal Parameters and Reproducibility

import random
import numpy as np
import torch

SID4 = 6812
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("SID4  =", SID4)
print("SEED  =", SEED)
print("SLICE =", SLICE)
print("HP_ID =", HP_ID)
print("CLS_A =", CLS_A)
print("CLS_B =", CLS_B)
print("\nDevice:", "cuda" if torch.cuda.is_available() else "cpu")

SID4  = 6812
SEED  = 6812
SLICE = 812
HP_ID = 2
CLS_A = 2
CLS_B = 9

Device: cuda


In [2]:
# Install required libraries.
# torchao is not required for this homework; remove an incompatible
# preinstalled version if Colab happens to provide one.
!pip uninstall -y torchao -q
!pip install -q -U transformers datasets peft accelerate sentencepiece evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 145.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00


In [3]:
import os
import random
import numpy as np
import torch

from torch.utils.data import DataLoader
from torch.optim import AdamW

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, TaskType, get_peft_model

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA L4


## Task 1 — Data Preparation



In [4]:
dataset = load_dataset("neil-code/dialogsum-test")

print(dataset)
print("\nAvailable splits:", dataset.keys())
print("\nTrain columns:", dataset["train"].column_names)
print("\nFirst raw example:")
print(dataset["train"][0])

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/441k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})

Available splits: dict_keys(['train', 'validation', 'test'])

Train columns: ['id', 'dialogue', 'summary', 'topic']

First raw example:
{'id': 'train_0', 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let 

In [5]:
def process_example(example):
    return {
        "input": "summarize: " + example["dialogue"],
        "target": example["summary"]
    }

processed_dataset = dataset.map(process_example)

print("=" * 80)
print("PROCESSED SAMPLE 1")
print("=" * 80)
print("INPUT:")
print(processed_dataset["train"][0]["input"])
print("\nTARGET:")
print(processed_dataset["train"][0]["target"])

print("\n" + "=" * 80)
print("PROCESSED SAMPLE 2")
print("=" * 80)
print("INPUT:")
print(processed_dataset["train"][1]["input"])
print("\nTARGET:")
print(processed_dataset["train"][1]["target"])

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

PROCESSED SAMPLE 1
INPUT:
summarize: #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

TARGET:
Mr. Smith's getting a 

## Task 2 — Baseline Inference

Load `google/flan-t5-small` and run inference on two dialogues.  
The same two dialogues are reused after fine-tuning for a fair comparison.

In [6]:
MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
baseline_model.config.tie_word_embeddings = False
baseline_model = baseline_model.to(device)

print("Loaded model:", MODEL_NAME)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded model: google/flan-t5-small


In [7]:
# Select two reproducible examples using SLICE.

test_split = processed_dataset["test"] if "test" in processed_dataset else processed_dataset["train"]

sample_indices = [
    SLICE % len(test_split),
    (SLICE + 1) % len(test_split)
]

evaluation_examples = [test_split[i] for i in sample_indices]

print("Sample indices:", sample_indices)

for i, example in enumerate(evaluation_examples, 1):
    print("\n" + "=" * 80)
    print(f"DIALOGUE {i}")
    print("=" * 80)
    print(example["dialogue"])
    print("\nREFERENCE SUMMARY:")
    print(example["summary"])

Sample indices: [313, 314]

DIALOGUE 1
#Person1#: Have you ever been invited to a Chinese feast?
#Person2#: Sure. Beijingers are distinguished for their warmth and hospitality.
#Person1#: As far as you could see, is dining at a Chinese table a distinct experience? To what extent does it differ from that in your homeland?
#Person2#: To begin with, I have to use chopsticks.
#Person1#: But you look quite proficient in using them now.
#Person2#: Certainly. After all, I've been here for two years. It's not much uphill work for a foreigner to get used to Chinese dinner utensils. Though, I do feel hazy about how to behave appropriately at a Chinese table.
#Person1#: Yes, table etiquette. There are a multitude of rules that foreigners might never know until they were told. Some practices even appear somewhat awkward or rude to Westerners, while they are the norm in China. Hey! Your chopsticks, old silly! You should never stab your chopsticks into the rice bowl!
#Person2#: Anything wrong with t

In [8]:
def generate_summary(model, dialogue):
    prompt = "summarize: " + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

baseline_outputs = []

for i, example in enumerate(evaluation_examples, 1):
    output = generate_summary(baseline_model, example["dialogue"])
    baseline_outputs.append(output)

    print("\n" + "=" * 80)
    print(f"BASELINE OUTPUT {i}")
    print("=" * 80)
    print(output)
    print("\nREFERENCE:")
    print(example["summary"])


BASELINE OUTPUT 1
You should never stab your chopsticks into the rice bowl.

REFERENCE:
#Person1# and #Person2# are discussing the differences between China and Western feasts. There are so many rules on the Chinese table, and they both feel hazy about its etiquette.

BASELINE OUTPUT 2
You should never stab your chopsticks into the rice bowl.

REFERENCE:
#Person1# and #Person2# talk about the difference in table etiquette in China. They both feel hazy about Chinese table etiquette and wrong use of chopsticks can lead to people's enrage.


## Prepare Training Data

A small subset is used so that the experiment is practical on Google Colab.  
The same subset and training settings are used for both LoRA ranks.

In [9]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

def tokenize_function(batch):
    model_inputs = tokenizer(
        batch["input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    # Keep real target token IDs here. DataCollatorForSeq2Seq will pad
    # label sequences with -100 so padded positions are ignored by loss.
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_data = processed_dataset["train"]
TRAIN_SIZE = min(1000, len(train_data))

# Reproducible contiguous subset beginning at the student's SLICE.
if SLICE + TRAIN_SIZE <= len(train_data):
    train_indices = list(range(SLICE, SLICE + TRAIN_SIZE))
else:
    train_indices = [(SLICE + i) % len(train_data) for i in range(TRAIN_SIZE)]

train_subset = train_data.select(train_indices)

tokenized_train = train_subset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_subset.column_names
)

print("Training examples:", len(tokenized_train))
print(tokenized_train)

# Sanity checks: every training example must contain target tokens.
label_lengths = [len(x["labels"]) for x in tokenized_train]
assert len(tokenized_train) > 0
assert min(label_lengths) > 0, "Found an example with empty labels."

print("Minimum target-token length:", min(label_lengths))
print("Maximum target-token length:", max(label_lengths))
print("First label token IDs:", tokenized_train[0]["labels"][:20])
print("Decoded first target:")
print(tokenizer.decode(tokenized_train[0]["labels"], skip_special_tokens=True))

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training examples: 1000
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})
Minimum target-token length: 11
Maximum target-token length: 121
First label token IDs: [1713, 345, 13515, 536, 4663, 6490, 79, 805, 378, 44, 3, 8689, 3253, 5, 6664, 9, 1390, 12, 3170, 4876]
Decoded first target:
#Person1# suggests they buy things at specialized stores. Masa plans to rent movies, and Masa will go to the mall with #Person1# and #Person3#.


## Tasks 3 & 4 — Apply LoRA and Fine-Tune

We train two LoRA adapters:
- Experiment 1: `r=4`
- Experiment 2: `r=16`

Other LoRA/training settings are kept fixed so rank is the main experimental change.

### Training implementation note

This notebook uses an explicit PyTorch training loop for the LoRA adapters.
The loss is read directly from `model(**batch).loss`, checked for validity,
backpropagated, and logged throughout training. This also makes the reported
final training loss easy to verify from the notebook output.

In [10]:
def parameter_counts(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    return total_params, trainable_params


def train_lora_model(rank, output_dir):
    # Reset seeds so r=4 and r=16 use reproducible data ordering.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # Fresh pretrained model for each LoRA-rank experiment.
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.config.tie_word_embeddings = False

    lora_config = LoraConfig(
        r=rank,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q", "v"],
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM
    )

    model = get_peft_model(model, lora_config)
    model = model.to(device)

    total, trainable = parameter_counts(model)

    print("=" * 80)
    print(f"LoRA CONFIGURATION — r={rank}")
    print("=" * 80)
    print("Rank (r):", rank)
    print("Alpha:", 32)
    print("Dropout:", 0.05)
    print("Target modules:", ["q", "v"])
    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"Trainable percentage: {100 * trainable / total:.4f}%")

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100,
        return_tensors="pt"
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        tokenized_train,
        batch_size=8,
        shuffle=True,
        collate_fn=data_collator,
        generator=generator
    )

    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=2e-4
    )

    # -------- Pre-training loss sanity check --------
    model.train()
    first_batch = next(iter(train_loader))
    first_batch = {k: v.to(device) for k, v in first_batch.items()}

    with torch.no_grad():
        initial_loss = model(**first_batch).loss.item()

    print(f"Initial batch loss: {initial_loss:.4f}")

    if not np.isfinite(initial_loss) or initial_loss <= 0:
        raise RuntimeError(
            f"Invalid initial loss ({initial_loss}). "
            "Training has been stopped instead of reporting a false 0.0 loss."
        )

    # -------- LoRA fine-tuning --------
    NUM_EPOCHS = 1
    global_step = 0
    loss_history = []

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0

        for step, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad(set_to_none=True)

            outputs = model(**batch)
            loss = outputs.loss

            if loss is None or not torch.isfinite(loss):
                raise RuntimeError(
                    f"Invalid training loss at epoch {epoch + 1}, step {step}: {loss}"
                )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0
            )
            optimizer.step()

            loss_value = loss.detach().float().item()
            running_loss += loss_value
            loss_history.append(loss_value)
            global_step += 1

            if step == 1 or step % 20 == 0 or step == len(train_loader):
                avg_so_far = running_loss / step
                print(
                    f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
                    f"Step {step}/{len(train_loader)} | "
                    f"Batch loss: {loss_value:.4f} | "
                    f"Average loss: {avg_so_far:.4f}"
                )

        epoch_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch + 1} average training loss: {epoch_loss:.4f}")

    final_training_loss = float(np.mean(loss_history))

    if not np.isfinite(final_training_loss) or final_training_loss <= 0:
        raise RuntimeError(
            f"Invalid final training loss: {final_training_loss}"
        )

    metrics = {
        "rank": rank,
        "initial_batch_loss": initial_loss,
        "final_average_train_loss": final_training_loss,
        "last_batch_loss": loss_history[-1],
        "train_steps": global_step,
        "train_examples": len(tokenized_train),
        "epochs": NUM_EPOCHS
    }

    print("\nFinal training metrics:")
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.6f}")
        else:
            print(f"{key}: {value}")

    # Save the LoRA adapter and tokenizer.
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Saved LoRA adapter to: {output_dir}")

    return model, metrics

### Experiment 1 — LoRA rank r=4

In [11]:
model_r4, metrics_r4 = train_lora_model(
    rank=4,
    output_dir="./lora_dialogsum_r4"
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LoRA CONFIGURATION — r=4
Rank (r): 4
Alpha: 32
Dropout: 0.05
Target modules: ['q', 'v']
Total parameters: 77,133,184
Trainable parameters: 172,032
Trainable percentage: 0.2230%
Initial batch loss: 2.7374
Epoch 1/1 | Step 1/125 | Batch loss: 2.5228 | Average loss: 2.5228
Epoch 1/1 | Step 20/125 | Batch loss: 1.9372 | Average loss: 2.2948
Epoch 1/1 | Step 40/125 | Batch loss: 1.6857 | Average loss: 2.0754
Epoch 1/1 | Step 60/125 | Batch loss: 1.7562 | Average loss: 1.9674
Epoch 1/1 | Step 80/125 | Batch loss: 1.5139 | Average loss: 1.8964
Epoch 1/1 | Step 100/125 | Batch loss: 1.6088 | Average loss: 1.8490
Epoch 1/1 | Step 120/125 | Batch loss: 1.9969 | Average loss: 1.8289
Epoch 1/1 | Step 125/125 | Batch loss: 1.7049 | Average loss: 1.8197
Epoch 1 average training loss: 1.8197

Final training metrics:
rank: 4
initial_batch_loss: 2.737375
final_average_train_loss: 1.819718
last_batch_loss: 1.704858
train_steps: 125
train_examples: 1000
epochs: 1
Saved LoRA adapter to: ./lora_dialogsum_r

## Task 5 — Post-Fine-Tuning Inference (r=4)

In [12]:
model_r4 = model_r4.to(device)

r4_outputs = []

for i, example in enumerate(evaluation_examples, 1):
    output = generate_summary(model_r4, example["dialogue"])
    r4_outputs.append(output)

    print("\n" + "=" * 80)
    print(f"LoRA r=4 OUTPUT {i}")
    print("=" * 80)
    print(output)
    print("\nREFERENCE:")
    print(example["summary"])


LoRA r=4 OUTPUT 1
#Person1# and #Person2# have been invited to a Chinese feast. #Person1# and #Person2# feel hazy about how to behave appropriately at a Chinese table.

REFERENCE:
#Person1# and #Person2# are discussing the differences between China and Western feasts. There are so many rules on the Chinese table, and they both feel hazy about its etiquette.

LoRA r=4 OUTPUT 2
#Person1# and #Person2# have been invited to a Chinese feast. #Person1# and #Person2# feel hazy about how to behave appropriately at a Chinese table.

REFERENCE:
#Person1# and #Person2# talk about the difference in table etiquette in China. They both feel hazy about Chinese table etiquette and wrong use of chopsticks can lead to people's enrage.


In [13]:
# Free the first trained model before training r=16 to reduce GPU memory use.

model_r4 = model_r4.to("cpu")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Experiment 2 — LoRA rank r=16

In [14]:
model_r16, metrics_r16 = train_lora_model(
    rank=16,
    output_dir="./lora_dialogsum_r16"
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LoRA CONFIGURATION — r=16
Rank (r): 16
Alpha: 32
Dropout: 0.05
Target modules: ['q', 'v']
Total parameters: 77,649,280
Trainable parameters: 688,128
Trainable percentage: 0.8862%
Initial batch loss: 2.7374
Epoch 1/1 | Step 1/125 | Batch loss: 2.5228 | Average loss: 2.5228
Epoch 1/1 | Step 20/125 | Batch loss: 1.9391 | Average loss: 2.2995
Epoch 1/1 | Step 40/125 | Batch loss: 1.6921 | Average loss: 2.0775
Epoch 1/1 | Step 60/125 | Batch loss: 1.7496 | Average loss: 1.9692
Epoch 1/1 | Step 80/125 | Batch loss: 1.5056 | Average loss: 1.8983
Epoch 1/1 | Step 100/125 | Batch loss: 1.6102 | Average loss: 1.8515
Epoch 1/1 | Step 120/125 | Batch loss: 2.0075 | Average loss: 1.8319
Epoch 1/1 | Step 125/125 | Batch loss: 1.7010 | Average loss: 1.8225
Epoch 1 average training loss: 1.8225

Final training metrics:
rank: 16
initial_batch_loss: 2.737375
final_average_train_loss: 1.822534
last_batch_loss: 1.701030
train_steps: 125
train_examples: 1000
epochs: 1
Saved LoRA adapter to: ./lora_dialogsu

In [15]:
model_r16 = model_r16.to(device)

r16_outputs = []

for i, example in enumerate(evaluation_examples, 1):
    output = generate_summary(model_r16, example["dialogue"])
    r16_outputs.append(output)

    print("\n" + "=" * 80)
    print(f"LoRA r=16 OUTPUT {i}")
    print("=" * 80)
    print(output)
    print("\nREFERENCE:")
    print(example["summary"])


LoRA r=16 OUTPUT 1
#Person1# and #Person2# have been invited to a Chinese feast. #Person1# and #Person2# feel hazy about how to behave at a Chinese table.

REFERENCE:
#Person1# and #Person2# are discussing the differences between China and Western feasts. There are so many rules on the Chinese table, and they both feel hazy about its etiquette.

LoRA r=16 OUTPUT 2
#Person1# and #Person2# have been invited to a Chinese feast. #Person1# and #Person2# feel hazy about how to behave at a Chinese table.

REFERENCE:
#Person1# and #Person2# talk about the difference in table etiquette in China. They both feel hazy about Chinese table etiquette and wrong use of chopsticks can lead to people's enrage.


## Task 6 — Output Comparison

In [16]:
for i, example in enumerate(evaluation_examples):
    print("\n" + "=" * 100)
    print(f"EXAMPLE {i + 1}")
    print("=" * 100)

    print("\nREFERENCE SUMMARY:")
    print(example["summary"])

    print("\nBASELINE:")
    print(baseline_outputs[i])

    print("\nLoRA r=4:")
    print(r4_outputs[i])

    print("\nLoRA r=16:")
    print(r16_outputs[i])


EXAMPLE 1

REFERENCE SUMMARY:
#Person1# and #Person2# are discussing the differences between China and Western feasts. There are so many rules on the Chinese table, and they both feel hazy about its etiquette.

BASELINE:
You should never stab your chopsticks into the rice bowl.

LoRA r=4:
#Person1# and #Person2# have been invited to a Chinese feast. #Person1# and #Person2# feel hazy about how to behave appropriately at a Chinese table.

LoRA r=16:
#Person1# and #Person2# have been invited to a Chinese feast. #Person1# and #Person2# feel hazy about how to behave at a Chinese table.

EXAMPLE 2

REFERENCE SUMMARY:
#Person1# and #Person2# talk about the difference in table etiquette in China. They both feel hazy about Chinese table etiquette and wrong use of chopsticks can lead to people's enrage.

BASELINE:
You should never stab your chopsticks into the rice bowl.

LoRA r=4:
#Person1# and #Person2# have been invited to a Chinese feast. #Person1# and #Person2# feel hazy about how to behav

### Comparison Discussion

**Write this section after inspecting your actual outputs.**

In 2–3 lines, describe the differences you actually observe between the baseline and fine-tuned summaries. Consider whether the fine-tuned models are more concise, capture the main point better, omit important information, or introduce incorrect details.

Do **not** claim that fine-tuning improved the model unless your generated outputs support that conclusion.

## Task 7 — LoRA Rank Experiment

In [17]:
print("=" * 80)
print("TRAINING METRICS")
print("=" * 80)

print("\nr=4:")
for key, value in metrics_r4.items():
    print(f"{key}: {value}")

print("\nr=16:")
for key, value in metrics_r16.items():
    print(f"{key}: {value}")

print("\nLoss comparison:")
print(f"r=4  average training loss : {metrics_r4['final_average_train_loss']:.4f}")
print(f"r=16 average training loss : {metrics_r16['final_average_train_loss']:.4f}")

TRAINING METRICS

r=4:
rank: 4
initial_batch_loss: 2.737374782562256
final_average_train_loss: 1.8197182064056396
last_batch_loss: 1.7048577070236206
train_steps: 125
train_examples: 1000
epochs: 1

r=16:
rank: 16
initial_batch_loss: 2.737374782562256
final_average_train_loss: 1.8225344657897948
last_batch_loss: 1.7010300159454346
train_steps: 125
train_examples: 1000
epochs: 1

Loss comparison:
r=4  average training loss : 1.8197
r=16 average training loss : 1.8225
